# WBerious Bridge — PASS-1 Recall Layer

This notebook contains the **corpus-level recall bridge** between the legal corpus and the FATE / WBerious precision stack.

---

## Purpose

The WBerious Bridge performs large-scale X-matching across a normalized legal corpus.

It is a **recall stage**, not a precision stage.

Its role is to identify candidate appeals that potentially support one or more X-tests defined in `Y_inferred.json`.

---

## Stage 1 — PASS-1 Corpus Scan

**Inputs:**

* `Y_inferred.json` (must contain `x_tests` definitions)
* Normalized appeal JSONL files (with `reasoning_for_index` or `summary` fields)

**Process:**

* For each appeal record:

  * Select `reasoning_for_index` (fallback: `summary`)
  * Send text to the LLM with a strict JSON schema request
  * Require X-matches to include character-span anchors (`start`, `end`)
  * Apply deterministic validation rules:

    * `matched_X` must be subset of allowed X IDs
    * Anchors must be in-bounds spans
    * Each X must have at least one anchor (unless explicitly disabled)
  * Derive verbatim `evidence_snippets` directly from spans (no model-trusted substrings)

**Output:**

* `pass1_results.jsonl`
* Append-only and resume-safe
* Each row contains:

  * `matched_X`
  * span anchors
  * derived `evidence_snippets`
  * confidence (0–100)
  * metadata

This stage enables scalable candidate discovery across thousands of appeals.

---

## Stage 2 — Shortlist Export

This step:

* Loads `pass1_results.jsonl`
* Maps X IDs to human-readable names from `Y`
* Flattens evidence snippets for inspection
* Filters to appeals with any matched X
* Sorts by confidence
* Exports `matched_x_results.csv`

This produces a ranked shortlist of appeals for deeper review.

---

## Architectural Position

PASS-1 = **Recall Layer**
Moltie = **Precision + Paragraph-Anchored Interrogation Layer**

The Bridge does not replace Moltie.

It identifies which cases are worth sending to Moltie for strict para-level anchoring and evidence validation.

---

## Guid


In [ ]:
from pathlib import Path
import subprocess, shlex
import tqdm

NB_DIR = Path.cwd()
OUT_DIR = NB_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# If your Y exists in Statements/output, point to it:
Y_SOURCE = Path("/home/hello/Projects/Statements/output/Y_inferred.json")

# Place a local copy next to the notebook (so paths stay notebook-relative)
Y_JSON = OUT_DIR / "Y_inferred.json"
if not Y_JSON.exists():
    Y_JSON.write_text(Y_SOURCE.read_text(encoding="utf-8"), encoding="utf-8")

PASS1_OUT = OUT_DIR / "pass1_results.jsonl"

BASE_DIR = Path("/home/hello/Projects/FATE_Dr_WB/Dr_WBerious/output/legal-corpus/normalized")
INPUT_FILES = [
    BASE_DIR / "judgments_claimant_fav_faiss_chunk_with_reasoning.jsonl",
    BASE_DIR / "judgments_respondent_fav_faiss_chunk_with_reasoning.jsonl",
    BASE_DIR / "judgments_unknown_faiss_chunk_with_reasoning.jsonl",
]

import shlex
from pathlib import Path

PASS1_SCRIPT = Path("/home/hello/Projects/Statements/code/pass1_scan.py")

cmd = [
    "python", str(PASS1_SCRIPT),
    "--y-json", str(Y_JSON),
    "--out", str(PASS1_OUT),
    "--model", "mistral-small3.2:latest",
    "--ollama-url", "http://localhost:11434/api/generate",
    #"--debug-max", "200",
]

cmd += ["--input", *map(str, INPUT_FILES)]  # <-- key change

cmd_str = " ".join(shlex.quote(c) for c in cmd)
print("Running:\n", cmd_str)
!{cmd_str}



In [ ]:
import json
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd()
OUT_DIR = BASE_DIR / "output"
PASS1_PATH = OUT_DIR / "pass1_results.jsonl"
Y_PATH = OUT_DIR / "Y_inferred.json"

# --- load pass1 ---
rows = []
with PASS1_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
df = pd.DataFrame(rows)

# --- load Y (for X-name mapping) ---
Y = json.loads(Y_PATH.read_text(encoding="utf-8"))
x_tests = (Y.get("x_tests") or {})
X_NAME = {k: (v.get("name") or "").strip() for k, v in x_tests.items()}

def matched_names(matched):
    if not isinstance(matched, list):
        return ""
    out = []
    for x in matched:
        x = str(x).strip()
        if not x:
            continue
        name = X_NAME.get(x, "")
        out.append(f"{x}: {name}" if name else x)
    return " | ".join(out)

df["matched_X_names"] = df["matched_X"].apply(matched_names) if "matched_X" in df.columns else ""

# filename guess (same logic)
def filename_guess(row):
    fn = row.get("filename")
    if isinstance(fn, str) and fn.strip():
        return fn.strip()
    item_id = row.get("item_id")
    if not isinstance(item_id, str):
        return None
    parts = item_id.split("::")
    return parts[2] if len(parts) >= 3 else None

df["filename_guess"] = df.apply(filename_guess, axis=1)

# flatten evidence snippets into a single readable field
def flatten_snips(snips, max_each=240, max_total=1200):
    if not isinstance(snips, list):
        return ""
    chunks = []
    total = 0
    for s in snips:
        if not isinstance(s, dict):
            continue
        x = str(s.get("x") or "").strip()
        txt = str(s.get("snippet") or "").strip()
        if not txt:
            continue
        txt = txt.replace("\n", " ")
        if len(txt) > max_each:
            txt = txt[:max_each] + "…"
        chunk = f"{x}: {txt}" if x else txt
        if total + len(chunk) > max_total:
            chunks.append("…")
            break
        chunks.append(chunk)
        total += len(chunk)
    return " | ".join(chunks)

df["evidence_snips_flat"] = df["evidence_snippets"].apply(flatten_snips) if "evidence_snippets" in df.columns else ""

# focus: rows with ANY match
matched_df = df[df["matched_X"].apply(lambda x: isinstance(x, list) and len(x) > 0)].copy() if "matched_X" in df.columns else df.iloc[0:0].copy()
matched_df = matched_df.sort_values("confidence", ascending=False) if "confidence" in matched_df.columns else matched_df

# columns to export/view (only keep what exists)
cols = [
    "filename_guess",
    "appeal_type",
    "who_appealed",
    "outcome",
    "successful",
    "summary_clean",
    "reasoning_for_index",
    "confidence",
    "matched_X",
    "matched_X_names",
    "evidence_snips_flat",
    "note",
    "source_file",
    "source_field",
    "item_id",
]
cols = [c for c in cols if c in matched_df.columns]

# export
matched_df.to_csv(OUT_DIR / "matched_x_results.csv", columns=cols, index=False)

# view
matched_df.head(10)[cols]
